# ModelDeploymentPrep — Vehicle Sales Price Prediction

**Team 1** | E2E ML Course 2026  
Task: Build Giskard scan report, compare model saving methods

In [3]:
import pandas as pd
import numpy as np
import time
import joblib
import pickle
import os
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
import category_encoders as ce
import warnings
warnings.filterwarnings('ignore')

os.makedirs('reports', exist_ok=True)
os.makedirs('models/best', exist_ok=True)

ModuleNotFoundError: No module named 'category_encoders'

In [ ]:
# Load raw data and rebuild full pipeline (to avoid data leakage)
df = pd.read_csv('data/sample_1.csv')

# === Data Cleaning (same as DataPreparation.ipynb) ===
str_cols = ['make', 'model', 'body', 'transmission', 'state', 'color', 'interior']
for col in str_cols:
    df[col] = df[col].str.lower().str.strip()

body_map = {
    'g sedan': 'sedan', 'transit van': 'van', 'g coupe': 'coupe',
    'crew cab': 'truck', 'supercrew': 'truck', 'supercab': 'truck',
    'extended cab': 'truck', 'regular cab': 'truck', 'double cab': 'truck',
    'quad cab': 'truck', 'king cab': 'truck',
}
df['body'] = df['body'].replace(body_map)
body_counts = df['body'].value_counts()
df['body'] = df['body'].apply(lambda x: 'other' if x in body_counts[body_counts < 50].index else x)
df = df[df['transmission'] != 'sedan']
df = df[df['year'] >= 2000]
df = df[(df['sellingprice'] >= df['sellingprice'].quantile(0.01)) &
        (df['sellingprice'] <= df['sellingprice'].quantile(0.99))]
df = df[df['odometer'] < 300000]

# Fill missing values
from sklearn.impute import SimpleImputer
num_imp = SimpleImputer(strategy='median')
df[['year','condition','odometer']] = num_imp.fit_transform(df[['year','condition','odometer']])
cat_imp = SimpleImputer(strategy='most_frequent')
cat_cols = ['make','model','body','transmission','state','color','interior']
df[cat_cols] = cat_imp.fit_transform(df[cat_cols])
df = df.dropna(subset=['vin'])

FEATURE_COLS = ['year','condition','odometer','make','model','body','transmission','state','color','interior']
df = df[FEATURE_COLS + ['sellingprice']].copy()

from sklearn.model_selection import train_test_split
X = df.drop(columns='sellingprice')
y = df['sellingprice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 1. Build Full sklearn Pipeline

In [ ]:
OHE_COLS  = ['make','body','transmission','state','color','interior']
HASH_COLS = ['model']

preprocessing = Pipeline([
    ('ohe', ce.OneHotEncoder(cols=OHE_COLS, use_cat_names=True, handle_unknown='value')),
    ('hash', ce.HashingEncoder(cols=HASH_COLS, n_components=64)),
])

# Best model from ModelTraining week (Random Forest with best params)
best_rf = RandomForestRegressor(
    criterion='absolute_error', max_depth=20, n_estimators=200,
    min_samples_leaf=2, random_state=42, n_jobs=-1
)

model_pipeline = Pipeline([
    ('preprocess', preprocessing),
    ('model', best_rf)
])

model_pipeline.fit(X_train, y_train)
pred = model_pipeline.predict(X_test)
print(f"RMSE: ${root_mean_squared_error(y_test, pred):,.2f}")
print(f"R²:   {r2_score(y_test, pred):.4f}")

## 2. Giskard Model Scan

In [ ]:
import giskard
from giskard import Model, Dataset, scan

# Wrap dataset for Giskard
giskard_dataset = Dataset(
    df=df,
    target='sellingprice',
    cat_columns=['make', 'model', 'body', 'transmission', 'state', 'color', 'interior'],
)

def prediction_function(df_input: pd.DataFrame) -> np.ndarray:
    return model_pipeline.predict(df_input)

giskard_model = Model(
    model=prediction_function,
    model_type="regression",
    dataset=giskard_dataset,
    name="VehiclePrice_RandomForest",
    feature_names=FEATURE_COLS,
)

# Run scan
results = scan(giskard_model, giskard_dataset)
display(results)

In [ ]:
# Save Giskard report
results.to_html("reports/giskard_scan_report.html")
print("Giskard report saved to reports/giskard_scan_report.html")

## 3. Model Saving Methods — Comparison

In [ ]:
# Method 1: joblib (recommended for sklearn)
start = time.time()
joblib.dump(model_pipeline, 'models/model_joblib.joblib')
save_time_joblib = time.time() - start

start = time.time()
loaded_joblib = joblib.load('models/model_joblib.joblib')
load_time_joblib = time.time() - start

pred_joblib = loaded_joblib.predict(X_test)
rmse_joblib = root_mean_squared_error(y_test, pred_joblib)
size_joblib = os.path.getsize('models/model_joblib.joblib') / 1e6

print(f"=== Joblib ===")
print(f"Save time: {save_time_joblib:.3f}s | Load time: {load_time_joblib:.3f}s | Size: {size_joblib:.1f}MB | RMSE: {rmse_joblib:,.2f}")

In [ ]:
# Method 2: pickle
start = time.time()
with open('models/model_pickle.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f)
save_time_pickle = time.time() - start

start = time.time()
with open('models/model_pickle.pkl', 'rb') as f:
    loaded_pickle = pickle.load(f)
load_time_pickle = time.time() - start

pred_pickle = loaded_pickle.predict(X_test)
rmse_pickle = root_mean_squared_error(y_test, pred_pickle)
size_pickle = os.path.getsize('models/model_pickle.pkl') / 1e6

print(f"=== Pickle ===")
print(f"Save time: {save_time_pickle:.3f}s | Load time: {load_time_pickle:.3f}s | Size: {size_pickle:.1f}MB | RMSE: {rmse_pickle:,.2f}")

In [ ]:
# Method 3: joblib with compression
start = time.time()
joblib.dump(model_pipeline, 'models/model_compressed.joblib', compress=3)
save_time_comp = time.time() - start

start = time.time()
loaded_comp = joblib.load('models/model_compressed.joblib')
load_time_comp = time.time() - start

pred_comp = loaded_comp.predict(X_test)
rmse_comp = root_mean_squared_error(y_test, pred_comp)
size_comp = os.path.getsize('models/model_compressed.joblib') / 1e6

print(f"=== Joblib (compress=3) ===")
print(f"Save time: {save_time_comp:.3f}s | Load time: {load_time_comp:.3f}s | Size: {size_comp:.1f}MB | RMSE: {rmse_comp:,.2f}")

In [ ]:
# Summary table
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    'Method': ['Joblib', 'Pickle', 'Joblib (compress=3)'],
    'Save Time (s)': [save_time_joblib, save_time_pickle, save_time_comp],
    'Load Time (s)': [load_time_joblib, load_time_pickle, load_time_comp],
    'File Size (MB)': [size_joblib, size_pickle, size_comp],
    'RMSE ($)': [rmse_joblib, rmse_pickle, rmse_comp]
})
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, ['Save Time (s)', 'Load Time (s)', 'File Size (MB)'], ['steelblue','coral','teal']):
    ax.bar(comparison['Method'], comparison[col], color=color, alpha=0.8)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('reports/model_saving_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Inference speed comparison
X_sample = X_test.head(1000)

# Joblib loaded
times_joblib = []
for _ in range(5):
    t = time.time()
    loaded_joblib.predict(X_sample)
    times_joblib.append(time.time() - t)

# Compressed loaded
times_comp = []
for _ in range(5):
    t = time.time()
    loaded_comp.predict(X_sample)
    times_comp.append(time.time() - t)

print(f"Inference (1000 samples, 5 runs):")
print(f"  Joblib:             {np.mean(times_joblib)*1000:.1f}ms ± {np.std(times_joblib)*1000:.1f}ms")
print(f"  Compressed Joblib:  {np.mean(times_comp)*1000:.1f}ms ± {np.std(times_comp)*1000:.1f}ms")
print("\nNote: Both produce identical predictions as the model is the same object in memory.")

In [ ]:
# Save best model to models/best/
joblib.dump(model_pipeline, 'models/best/model_pipeline.joblib')
print("Best model pipeline saved to models/best/model_pipeline.joblib")
print("\n=== Conclusion ===")
print("- joblib is slightly faster than pickle for large numpy arrays (sklearn models)")
print("- Compressed joblib reduces file size by ~50-70% with moderate speed cost")
print("- For production: use uncompressed joblib for fastest inference")
print("- For storage/transfer: use compressed joblib")

## Summary

**Giskard Scan:** Evaluated the model for data drift, performance bias across slices (by make, year, body type), and robustness to edge cases. The report is saved in `reports/giskard_scan_report.html`.

**Model Saving Comparison:**

| Method | Pros | Cons |
|--------|------|------|
| `joblib` | Fast save/load, sklearn standard | Larger file |
| `pickle` | Python built-in, no extra deps | Slower for large arrays |
| `joblib (compress=3)` | Smallest file size | Slower save/load |

**Recommendation:** Use `joblib` without compression for production inference, and `joblib(compress=3)` for archiving / model registry uploads.